# Candidate Generation

## 1. Objective

The objective of this notebook is to build and evaluate a multi-source candidate generation layer for the recommendation system.

Previous experiments showed that:

- recent 7-day popularity is a strong non-personalised baseline;
- implicit ALS provides personalised recommendations but does not outperform recent popularity when directly evaluated at Top-12;
- different recommendation approaches capture different signals.

Rather than requiring one model to produce the final recommendations, this notebook treats these approaches as candidate generators.

The candidate generation stage aims to retrieve a relatively small set of potentially relevant articles for each customer. A downstream ranking model can later score these candidates and select the final Top-12 recommendations.

Initial candidate sources:

1. Recent popularity
2. ALS collaborative filtering
3. Item-item collaborative filtering

Candidate quality will primarily be evaluated using Recall@K.

In [2]:
import numpy as np
import pandas as pd

from scipy.sparse import csr_matrix

from implicit.als import AlternatingLeastSquares
from implicit.nearest_neighbours import CosineRecommender

### 2. Load Transactions

Only the fields required for recommendation modelling are loaded.

The same temporal evaluation framework used in the previous notebooks is retained so that results remain comparable.

In [3]:
transactions = pd.read_csv(
    "../data/raw/transactions_train.csv",
    usecols=["t_dat", "customer_id", "article_id"]
)

transactions["t_dat"] = pd.to_datetime(transactions["t_dat"])

transactions.shape

transactions["t_dat"].agg(["min", "max"])

min   2018-09-20
max   2020-09-22
Name: t_dat, dtype: datetime64[ns]